# Task 3 & 4: Model Development, Comparison & Risk Evaluation
## Fraud Risk Scoring Platform

This notebook demonstrates:
1. **Training 3 Diverse Classifiers with Imbalance Handling**
   - Baseline: Balanced Logistic Regression
   - Candidate 1: Random Forest Classifier (`class_weight='balanced_subsample'`)
   - Candidate 2: HistGradientBoosting (`class_weight='balanced'`)
2. **Hyperparameter Tuning & Cross-Validation**
3. **Rigorous Evaluation on Held-Out Test Data** (PR-AUC, ROC-AUC, Recall, Precision, F1)
4. **Why Accuracy is Misleading** in Rare-Event Fraud Detection
5. **Decision Threshold Optimization & Risk Band Calibration**
6. **Model Artifact Serialization (`fraud_model.joblib`)**

In [1]:
import os
import sys
import warnings
sys.path.append('..')

from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings('ignore', category=InconsistentVersionWarning)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})


## 1. Load Unseen Test Data & Trained Champion Pipeline
The evaluation is executed strictly on the unseen test partition (`data/processed/test_set.csv`).

In [2]:
df_test = pd.read_csv('../data/processed/test_set.csv')
y_test = df_test['is_fraud'].values
X_test = df_test.drop(columns=['is_fraud'])

print(f'Test Data Shape: {df_test.shape}')
print(f'Test Fraud Rate: {y_test.mean()*100:.3f}% ({y_test.sum()} positive cases)')

model_path = '../models/fraud_model.joblib'
champion_pipeline = joblib.load(model_path)
print('Loaded Champion Pipeline:', champion_pipeline)

## 2. Quantitative Model Comparison
Comparing the Baseline against Random Forest and HistGradientBoosting.

In [3]:
model_files = {
    'Logistic Regression (Baseline)': '../models/logistic_regression_baseline_pipeline.joblib',
    'Random Forest': '../models/random_forest_pipeline.joblib',
    'HistGradientBoosting': '../models/hist_gradient_boosting_pipeline.joblib'
}

results = []
test_predictions = {}

for name, path in model_files.items():
    pipe = joblib.load(path)
    probs = pipe.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    test_predictions[name] = probs
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1-Score': f1_score(y_test, preds, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probs),
        'PR-AUC': average_precision_score(y_test, probs)
    })

results_df = pd.DataFrame(results).sort_values(by='PR-AUC', ascending=False)
results_df

## 3. Confusion Matrix Analysis (Threshold = 0.50)
Evaluating True Positives, False Positives, and False Negatives.

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, probs) in zip(axes, test_predictions.items()):
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(f'{name}\n(p >= 0.50)', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 4. Precision-Recall & ROC Curves
Visualizing the superiority of tree-based ensembles across varying decision cutoffs.

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
for name, probs in test_predictions.items():
    p, r, _ = precision_recall_curve(y_test, probs)
    ax1.plot(r, p, label=f'{name} (PR-AUC={average_precision_score(y_test, probs):.3f})', linewidth=2)
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax2.plot(fpr, tpr, label=f'{name} (ROC-AUC={roc_auc_score(y_test, probs):.3f})', linewidth=2)

ax1.set_xlabel('Recall (Fraud Detection Rate)')
ax1.set_ylabel('Precision (Alert Accuracy)')
ax1.set_title('Precision-Recall Curves', fontweight='bold')
ax1.legend()

ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curves', fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.show()

## 5. Threshold Optimization & Multi-Tier Risk Bands
Rather than arbitrary binary decisions, financial risk systems categorize transactions into actionable operational tiers:
- **Low Risk / APPROVE ($p < 0.20$):** Frictionless authorization for ~98% of clean traffic.
- **Medium Risk / REVIEW ($0.20 \le p < 0.65$):** Step-up authentication (SMS OTP / 3DS challenge) without direct decline.
- **High Risk / DECLINE ($p \ge 0.65$):** Hard block to stop automated high-loss attacks.

In [6]:
rf_probs = test_predictions['Random Forest']
thresholds = np.linspace(0.05, 0.95, 50)
precs, recs, f1s = [], [], []

for t in thresholds:
    p = (rf_probs >= t).astype(int)
    precs.append(precision_score(y_test, p, zero_division=0))
    recs.append(recall_score(y_test, p, zero_division=0))
    f1s.append(f1_score(y_test, p, zero_division=0))

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precs, label='Precision', color='#2b5c8f', linewidth=2.5)
plt.plot(thresholds, recs, label='Recall', color='#d62728', linewidth=2.5)
plt.plot(thresholds, f1s, label='F1-Score', color='#2ca02c', linestyle='--', linewidth=2.5)
plt.axvspan(0.00, 0.20, color='green', alpha=0.1, label='APPROVE (p < 0.20)')
plt.axvspan(0.20, 0.65, color='orange', alpha=0.15, label='REVIEW (0.20 <= p < 0.65)')
plt.axvspan(0.65, 1.00, color='red', alpha=0.1, label='DECLINE (p >= 0.65)')
plt.xlabel('Probability Threshold')
plt.ylabel('Score')
plt.title('Threshold Trade-off Curves & Operational Risk Bands', fontweight='bold')
plt.legend(loc='center right')
plt.tight_layout()
plt.show()